In [ ]:
# DrugBank DDI Extraction Pipeline

This notebook extracts Drug-Drug Interactions (DDIs) from the DrugBank XML database, 
focusing on cardiovascular and antithrombotic drugs.

## Pipeline Steps:
1. Parse DrugBank XML database
2. Build ATC code lookup for drug classification  
3. Extract all drug-drug interactions
4. Filter for cardiovascular (ATC: C) and antithrombotic (ATC: B01) drugs
5. Export labeled dataset for severity classification

**Data Source:** DrugBank 5.x XML database  
**Output:** `data/ddi_cardio_or_antithrombotic_labeled.csv`

Mounted at /content/drive


In [ ]:
# ==============================================================================
# 1. IMPORTS AND CONFIGURATION
# ==============================================================================

import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

# Configuration
XML_PATH = "data/full database.xml"
OUTPUT_PATH = "data/ddi_cardio_or_antithrombotic_labeled.csv"
NS = {"db": "http://www.drugbank.ca"}  # DrugBank XML namespace

print(f"DrugBank XML: {XML_PATH}")
print(f"Output CSV: {OUTPUT_PATH}")

b'<?xml version="1.0" encoding="UTF-8"?>\n<drugbank xmlns="http://www.drugbank.ca" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.drugbank.ca http://www.drugbank.ca/'


In [ ]:
# ==============================================================================
# 2. PARSE DRUGBANK XML
# ==============================================================================

print("Loading DrugBank XML (this may take a few minutes)...")
tree = ET.parse(XML_PATH)
root = tree.getroot()

# Count total drugs
drug_count = len(root.findall("db:drug", NS))
print(f"Loaded {drug_count:,} drugs from DrugBank")

In [ ]:
# ==============================================================================
# 3. BUILD ATC CODE LOOKUP
# ==============================================================================
# ATC (Anatomical Therapeutic Chemical) Classification:
# - C: Cardiovascular system
# - B01: Antithrombotic agents

def classify_atc(atc_list):
    """Classify drug based on ATC codes."""
    if not atc_list:
        return False, False
    is_cardiovascular = any(code.startswith("C") for code in atc_list)
    is_antithrombotic = any(code.startswith("B01") for code in atc_list)
    return is_cardiovascular, is_antithrombotic

# Build lookup: drugbank_id -> list of ATC codes
drug_atc_lookup = {}

for drug in root.findall("db:drug", NS):
    drug_id_elem = drug.find("db:drugbank-id[@primary='true']", NS)
    if drug_id_elem is None:
        continue
    
    drug_id = drug_id_elem.text
    atc_list = [
        atc_elem.get("code")
        for atc_elem in drug.findall(".//db:atc-code", NS)
        if atc_elem.get("code")
    ]
    drug_atc_lookup[drug_id] = atc_list

print(f"Built ATC lookup for {len(drug_atc_lookup):,} drugs")

In [ ]:
# ==============================================================================
# 4. EXTRACT DRUG-DRUG INTERACTIONS
# ==============================================================================

rows = []

for drug in root.findall("db:drug", NS):
    # Source drug info
    src_id_elem = drug.find("db:drugbank-id[@primary='true']", NS)
    src_name_elem = drug.find("db:name", NS)
    
    if src_id_elem is None:
        continue
    
    src_id = src_id_elem.text
    src_name = src_name_elem.text if src_name_elem is not None else None
    atc_list_1 = drug_atc_lookup.get(src_id, [])
    is_cardio_1, is_antithrombotic_1 = classify_atc(atc_list_1)
    
    # Get interactions
    interactions = drug.find("db:drug-interactions", NS)
    if interactions is None:
        continue
    
    for interaction in interactions.findall("db:drug-interaction", NS):
        tgt_id_elem = interaction.find("db:drugbank-id", NS)
        tgt_name_elem = interaction.find("db:name", NS)
        description_elem = interaction.find("db:description", NS)
        
        if tgt_id_elem is None:
            continue
        
        tgt_id = tgt_id_elem.text
        tgt_name = tgt_name_elem.text if tgt_name_elem is not None else None
        atc_list_2 = drug_atc_lookup.get(tgt_id, [])
        is_cardio_2, is_antithrombotic_2 = classify_atc(atc_list_2)
        
        rows.append({
            "drugbank_id_1": src_id,
            "drug_name_1": src_name,
            "atc_1": atc_list_1,
            "is_cardiovascular_1": is_cardio_1,
            "is_antithrombotic_1": is_antithrombotic_1,
            "drugbank_id_2": tgt_id,
            "drug_name_2": tgt_name,
            "atc_2": atc_list_2,
            "is_cardiovascular_2": is_cardio_2,
            "is_antithrombotic_2": is_antithrombotic_2,
            "interaction_description": description_elem.text if description_elem is not None else None
        })

ddi_df = pd.DataFrame(rows)
print(f"Extracted {len(ddi_df):,} total drug-drug interactions")

,drugbank_id_1,drug_name_1,drugbank_id_2,drug_name_2,interaction_description
0,DB00001,Lepirudin,DB06605,Apixaban,Apixaban may increase the anticoagulant activi...
1,DB00001,Lepirudin,DB06695,Dabigatran etexilate,Dabigatran etexilate may increase the anticoag...
2,DB00001,Lepirudin,DB01254,Dasatinib,The risk or severity of bleeding and hemorrhag...
3,DB00001,Lepirudin,DB01609,Deferasirox,The risk or severity of gastrointestinal bleed...
4,DB00001,Lepirudin,DB01586,Ursodeoxycholic acid,The risk or severity of bleeding and bruising ...
...,...,...,...,...,...
2910005,DB31654,Influenza A virus A/Perth/722/2024 (H3N2) live...,DB13509,Aloxiprin,The risk or severity of Reye's syndrome can be...
2910006,DB31654,Influenza A virus A/Perth/722/2024 (H3N2) live...,DB13538,Guacetisal,The risk or severity of Reye's syndrome can be...
2910007,DB31654,Influenza A virus A/Perth/722/2024 (H3N2) live...,DB13612,Carbaspirin calcium,The risk or severity of Reye's syndrome can be...
2910008,DB31654,Influenza A virus A/Perth/722/2024 (H3N2) live...,DB14006,Choline salicylate,The risk or severity of Reye's syndrome can be...


In [ ]:
# ==============================================================================
# 5. FILTER FOR CARDIOVASCULAR/ANTITHROMBOTIC DRUGS
# ==============================================================================

# Keep interactions where at least one drug is cardiovascular OR antithrombotic
ddi_filtered = ddi_df[
    ddi_df["is_cardiovascular_1"] | 
    ddi_df["is_antithrombotic_1"] |
    ddi_df["is_cardiovascular_2"] | 
    ddi_df["is_antithrombotic_2"]
].copy()

print(f"Filtered to {len(ddi_filtered):,} cardiovascular/antithrombotic DDIs")
print(f"\nDataset summary:")
print(f"  - Unique drugs: {ddi_filtered['drug_name_1'].nunique() + ddi_filtered['drug_name_2'].nunique():,}")
print(f"  - Cardiovascular drug 1: {ddi_filtered['is_cardiovascular_1'].sum():,}")
print(f"  - Antithrombotic drug 1: {ddi_filtered['is_antithrombotic_1'].sum():,}")

,drugbank_id_1,drug_name_1,atc_1,is_cardiovascular_1,is_antithrombotic_1,drugbank_id_2,drug_name_2,atc_2,is_cardiovascular_2,is_antithrombotic_2,interaction_description
0,DB00001,Lepirudin,[B01AE02],False,True,DB06605,Apixaban,[B01AF02],False,True,Apixaban may increase the anticoagulant activi...
1,DB00001,Lepirudin,[B01AE02],False,True,DB06695,Dabigatran etexilate,[B01AE07],False,True,Dabigatran etexilate may increase the anticoag...
2,DB00001,Lepirudin,[B01AE02],False,True,DB01254,Dasatinib,[L01EA02],False,False,The risk or severity of bleeding and hemorrhag...
3,DB00001,Lepirudin,[B01AE02],False,True,DB01609,Deferasirox,[V03AC03],False,False,The risk or severity of gastrointestinal bleed...
4,DB00001,Lepirudin,[B01AE02],False,True,DB01586,Ursodeoxycholic acid,[A05AA02],False,False,The risk or severity of bleeding and bruising ...


In [ ]:
# ==============================================================================
# 6. PREVIEW DATA
# ==============================================================================

print("Sample interactions:")
ddi_filtered[["drug_name_1", "drug_name_2", "interaction_description"]].head(10)

Full interaction dataframe shape: (2910010, 11)


In [ ]:
# ==============================================================================
# 7. EXPORT DATASET
# ==============================================================================

# Save to CSV
ddi_filtered.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(ddi_filtered):,} DDIs to {OUTPUT_PATH}")

# Show final column info
print(f"\nColumns: {list(ddi_filtered.columns)}")

,drugbank_id_1,drug_name_1,atc_1,is_cardiovascular_1,is_antithrombotic_1,drugbank_id_2,drug_name_2,atc_2,is_cardiovascular_2,is_antithrombotic_2,interaction_description
0,DB00001,Lepirudin,[B01AE02],False,True,DB06605,Apixaban,[B01AF02],False,True,Apixaban may increase the anticoagulant activi...
1,DB00001,Lepirudin,[B01AE02],False,True,DB06695,Dabigatran etexilate,[B01AE07],False,True,Dabigatran etexilate may increase the anticoag...
2,DB00001,Lepirudin,[B01AE02],False,True,DB01254,Dasatinib,[L01EA02],False,False,The risk or severity of bleeding and hemorrhag...
3,DB00001,Lepirudin,[B01AE02],False,True,DB01609,Deferasirox,[V03AC03],False,False,The risk or severity of gastrointestinal bleed...
4,DB00001,Lepirudin,[B01AE02],False,True,DB01586,Ursodeoxycholic acid,[A05AA02],False,False,The risk or severity of bleeding and bruising ...
...,...,...,...,...,...,...,...,...,...,...,...
2909947,DB22790,Berahyaluronidase alfa,[],False,False,DB13378,Norfenefrine,[C01CA05],True,False,The risk or severity of adverse effects can be...
2909988,DB22790,Berahyaluronidase alfa,[],False,False,DB00695,Furosemide,"[C03EB01, C03CA01, G01AE10, C03CB01]",True,False,The therapeutic efficacy of Furosemide can be ...
2909996,DB31654,Influenza A virus A/Perth/722/2024 (H3N2) live...,[],False,False,DB00945,Acetylsalicylic acid,"[B01AC06, C07FX04, C10BX04, M01BA03, C10BX02, ...",True,True,The risk or severity of Reye's syndrome can be...
2910005,DB31654,Influenza A virus A/Perth/722/2024 (H3N2) live...,[],False,False,DB13509,Aloxiprin,"[N02BA02, B01AC15]",False,True,The risk or severity of Reye's syndrome can be...
